# A Batch Scripting Example for PhysIO Toolbox

**Author**: Kelly G. Garner, Michèle Masson-Trottier

<div style="line-height: 2;">
<a href="https://github.com/kel-github"><img src="https://img.shields.io/badge/-Kelly_G._Garner-181717?logo=github" alt="GitHub"></a><br>
<a href="https://github.com/micmas"><img src="https://img.shields.io/badge/-Michèle_Masson--Trottier-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-0642-5662"><img src="https://img.shields.io/badge/ORCID-0000--0002--0642--5662-green?logo=orcid" alt="ORCID"></a>
</div>

**Date**: 30/03/2026

**License:** 
<div style="margin-top: 10px;">
    <a href="https://creativecommons.org/licenses/by/4.0/" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> CC-BY-4.0 License
    </a>
</div>

## Purpose

Example of how to batch script the PhysIO Toolbox using Neurodesk to generate physiological regressors for multiple participants.

:::{admonition} Learning Objectives
:class: tip

After completing this tutorial, you will be able to:

- Create an example PhysIO batch script for a single participant using the Batch Editor GUI
- Generalise the script for multiple participants using a participant info structure
- Run the batch script from the Neurodesk PhysIO command-line container

:::

## Citation and Resources

**PhysIO Toolbox**
: Kasper, L., Bollmann, S., Diaconescu, A.O., Hutton, C., Heinzle, J., Iglesias, S., Hauser, T.U., Sebold, M., Manjaly, Z.-M., Pruessmann, K.P., Stephan, K.E., 2017. *The PhysIO Toolbox for Modeling Physiological Noise in fMRI Data*. Journal of Neuroscience Methods, 276, 56–72. https://doi.org/10.1016/j.jneumeth.2016.10.019

**TAPAS Software Package**
: Frässle, S., Aponte, E.A., Bollmann, S., Brodersen, K.H., Do, C.T., Harrison, O.K., Harrison, S.J., Heinzle, J., Iglesias, S., Kasper, L., Lomakina, E.I., Mathys, C., Müller-Schrader, M., Pereira, I., Petzschner, F.H., Raman, S., Schöbi, D., Toussaint, B., Weber, L.A., Yao, Y., Stephan, K.E., 2021. *TAPAS: an open-source software package for Translational Neuromodeling and Computational Psychiatry*. Frontiers in Psychiatry, 12, 680811. https://doi.org/10.3389/fpsyt.2021.680811

## Prerequisites

:::{admonition} Requirements
:class: warning

- Data largely in BIDS format
- `.log` files for physiological data (e.g., converted from CMRR `.zip` using [readCMRRPhysio.m](https://github.com/CMRR-C2P/MB/blob/master/readCMRRPhysio.m))
- Motion regressor files in `.txt` format compatible with SPM
- Running Neurodesk, PhysIO familiarity helpful

:::

## Section 1: Generate an Example Script for Batching

First, create an example batch script specific to one participant:

1. Download the relevant `.log` files for one participant and the `..._desc-confounds_timeseries.tsv` output from fmriprep
2. Convert motion parameters to `.txt` format (space-separated, compatible with SPM)
3. Follow the [PhysIO Quickstart](https://gitlab.ethz.ch/physio/physio-doc/-/wikis/QUICKSTART) to create a batch via the Batch Editor GUI

![PhysIO Batch Editor showing a participant-specific batch script](/static/tutorials/functional_imaging/PhysIO_Batch/PhysIOBatch1.png)
*An example participant-specific PhysIO batch script loaded in the SPM Batch Editor.*

## Section 2: Generalise the Script for Multiple Participants

**Step 1**: Create an `info` structure for each participant (saved as a `.mat` file):

```matlab
% info structure fields:
% sub_num    - subject number string, e.g. '01'
% sess       - session number (integer)
% nrun       - number of runs
% nscans     - number of scans per run [1 x nrun]
% cardiac_files     - cardiac log files {1 x nrun}
% respiration_files - respiration log files {1 x nrun}
% scan_timing       - scan timing file {1 x nrun}
% movement         - motion regressor .txt files {1 x nrun}
```

**Step 2**: Amend the batch script to load the `info` file and fill in the required fields:

```matlab
%% written by K. Garner, 2022
sub = '01';
dat_path = '/path/to/derivatives';
task = 'attlearn';
load(fullfile(dat_path, sprintf('sub-%s', sub), 'ses-02', 'func', ...
              sprintf('sub-%s_ses-02_task-%s_desc-physioinfo', sub, task)))

nrun = info.nrun;
nscans = info.nscans;
cardiac_files = info.cardiac_files;
respiration_files = info.respiration_files;
scan_timing = info.scan_timing;
movement = info.movement;

spm_jobman('initcfg');
spm('defaults', 'FMRI');

for irun = 1:nrun
    clear matlabbatch
    matlabbatch{1}.spm.tools.physio.save_dir = cellstr(fullfile(dat_path, ...
        sprintf('sub-%s', sub), 'ses-02', 'func'));
    matlabbatch{1}.spm.tools.physio.log_files.vendor = 'Siemens_Tics';
    matlabbatch{1}.spm.tools.physio.log_files.cardiac = cardiac_files(irun);
    matlabbatch{1}.spm.tools.physio.log_files.respiration = respiration_files(irun);
    matlabbatch{1}.spm.tools.physio.log_files.scan_timing = scan_timing(irun);
    matlabbatch{1}.spm.tools.physio.scan_timing.sqpar.Nslices = 81;
    matlabbatch{1}.spm.tools.physio.scan_timing.sqpar.TR = 1.51;
    matlabbatch{1}.spm.tools.physio.scan_timing.sqpar.Nscans = nscans(irun);
    matlabbatch{1}.spm.tools.physio.model.retroicor.yes.order.c = 3;
    matlabbatch{1}.spm.tools.physio.model.retroicor.yes.order.r = 4;
    matlabbatch{1}.spm.tools.physio.model.retroicor.yes.order.cr = 1;
    matlabbatch{1}.spm.tools.physio.model.movement.yes.file_realignment_parameters = ...
        {fullfile(dat_path, sprintf('sub-%s', sub), 'ses-02', 'func', ...
        sprintf('sub-%s_ses-02_task-%s_run-%d_desc-motion_timeseries.txt', sub, task, irun))};
    matlabbatch{1}.spm.tools.physio.model.movement.yes.order = 6;
    matlabbatch{1}.spm.tools.physio.verbose.level = 2;
    spm_jobman('run', matlabbatch);
end
```

## Section 3: Ready to Run on Neurodesk!

**Step 1**: Verify the script details are correct at the top. The script can easily be amended to run multiple subjects.

**Step 2**: On Neurodesk, go to **Functional Imaging → physio → physio r7771** (command-line tool, NOT the GUI interface).

![Selecting the PhysIO command-line container in Neurodesk](/static/tutorials/functional_imaging/PhysIO_Batch/PhysIOBatch2.png)
*Selecting the PhysIO command-line container (not the GUI) from the Neurodesk application menu.*

**Step 3**: Run your PhysIO batch script with:

```bash
run_spm12.sh /opt/mcr/v99/ batch /your/batch/script/named_something.m
```

Physiological regressors are now generated and ready for your GLM analysis!

## Summary

You have successfully:

- Created a participant-specific PhysIO batch script using the Batch Editor GUI
- Generalised the script for multiple participants using an info structure
- Ran the batch script from the Neurodesk command-line PhysIO container

This workflow enables automated, reproducible physiological noise correction across your entire study.

## See Also

- [PhysIO GUI Tutorial](./physio.ipynb) — Interactive introduction to PhysIO
- [SPM Tutorial](./spm.ipynb) — Incorporate these regressors into your GLM
- [PhysIO GitHub Repository](https://github.com/translationalneuromodeling/tapas) — Full documentation and examples